## Split Overmerged Authors — Phase 3: Credential-Only Profiles

Identifies work_authors attached to profiles whose `full_name` is composed entirely of credential acronyms (`DNP`, `FACOG`, `EdD EdD`, `MD MD`, `MD PhD`, …). These are catastrophic overmerges where source metadata (crossref/datacite) provided only a credential string instead of a name, and many unrelated works pooled into one credential bucket. No person identity is recoverable, so every attached work must be nulled and re-matched.

Target set built against the `_NP_CREDENTIAL_ACRONYMS` list fixed in `CreateAuthorNames`. After the parser re-run, **~456 profiles / ~1.8K work_authors rows / ~1.6K distinct works** survive — everything else was fixed at the parser layer.

**Candidate definition (profile must match AT LEAST ONE):**
- `lower(trim(full_name))` is exactly one credential acronym
- `first = ''` and `last` is a credential acronym
- `full_name` is a whitespace-joined sequence of credential acronyms only (e.g., `MD PhD`, `EdD EdD`)

**Safety filters:**
- Excludes profiles with any ORCID value (1 edge case — real person registered with only `MSc`)
- No other filters needed: if the profile name is pure credentials, there is no real person to preserve, and no co-signal (name match, institution, etc.) can validate the attachment

**Action:** null `author_id` on every work_authors row pointing to these profiles. No rollback cell — the entire population is uniformly unreliable, and there is no name to match against for a selective rollback.

**Follow-ups (out of scope for Phase 3):**
- ~188 RENAME-target profiles (e.g., `Mike Greenwood PhD`, `Robert E James III PhD (NETL)`) where the credential survived parsing but a real name is present. Needs parser extension for parenthesized suffixes (`PhD (C)`, `PhD(c)`), not a split.
- Downstream: raw_author_names that are themselves bare credentials (e.g., `raw = 'Facc'`) will not re-match productively in MatchAuthors. Those rows stay NULL, which is the correct outcome — source metadata is unrecoverable.

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_phase3 AS
WITH credential_profiles AS (
  SELECT afm.author_id, afm.full_name, afm.first, afm.last,
         afm.works_count, afm.orcid
  FROM openalex.authors.authors_for_matching afm
  WHERE (afm.orcid IS NULL OR afm.orcid = '')
    AND (
         lower(trim(afm.full_name)) IN (
           'phd','dphil','drph','scd','edd','psyd',
           'pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
           'msc','mph','mscn','msn','mpp','mhs',
           'llm','jd','lcsw',
           'facs','facp','facog','faap','fccp','faan',
           'frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
           'rn','crna','fnp',
           'md','do'
         )
      OR (
           afm.first = ''
           AND lower(afm.last) IN (
             'phd','dphil','drph','scd','edd','psyd',
             'pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
             'msc','mph','mscn','msn','mpp','mhs',
             'llm','jd','lcsw',
             'facs','facp','facog','faap','fccp','faan',
             'frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
             'rn','crna','fnp',
             'md','do'
           )
         )
      OR lower(trim(afm.full_name)) RLIKE
           '^(phd|dphil|drph|scd|edd|psyd|pharmd|mbbs|mbchb|dvm|dds|dmd|dnp|dpt|msc|mph|mscn|msn|mpp|mhs|llm|jd|lcsw|facs|facp|facog|faap|fccp|faan|frcp|frcs|frcpc|frcog|mrcp|mrcs|fcps|rn|crna|fnp|md|do)( (phd|dphil|drph|scd|edd|psyd|pharmd|mbbs|mbchb|dvm|dds|dmd|dnp|dpt|msc|mph|mscn|msn|mpp|mhs|llm|jd|lcsw|facs|facp|facog|faap|fccp|faan|frcp|frcs|frcpc|frcog|mrcp|mrcs|fcps|rn|crna|fnp|md|do))*$'
    )
)
SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
  cp.full_name AS author_full_name, cp.last AS author_last,
  cp.works_count
FROM openalex.works.work_authors wa
JOIN credential_profiles cp ON wa.author_id = cp.author_id
WHERE wa.author_id IS NOT NULL

### Cell 2: Validation statistics

In [ ]:
SELECT COUNT(*) AS total_candidates,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id) AS distinct_works,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates_phase3

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT author_id, author_full_name, author_last, works_count,
  raw_author_name, work_id
FROM openalex.authors.overmerge_split_candidates_phase3
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_phase3 AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, author_full_name, author_last,
  current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_phase3

### Cell 5: Execute the split

**WARNING**: This nulls out author_ids. Verify cells 2–3 before running.

**NOTE**: MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically. Same issue as Phase 1/2.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_phase3
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue split works for reprocessing by UpdateWorkAuthorships

Push affected work_ids into `curated_work_ids_pending_sync` so the next `UpdateWorkAuthorships` run picks them up and propagates the nulled author_ids through to `work_authorships` → `openalex_works`.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_phase3

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase3 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking (run after MatchAuthors re-processes)

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase3 log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1